In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Convert timestamps to timezone-aware datetimes in UTC+8 (Asia/Shanghai) per rule 10
tz = pytz.timezone('Asia/Shanghai')
for df in (metric, trace, log, error_logs):
    if 'timestamp' in df.columns:
        # timestamps are in seconds
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Helper to aggregate a dataframe with a kpi column name
def aggregate_kpi(df, component_col, kpi_col):
    # Ensure value is numeric
    df = df.copy()
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    group = df.groupby([component_col, kpi_col])['value']
    agg = group.agg(
        count='count',
        p50=lambda x: x.dropna().quantile(0.50),
        p95=lambda x: x.dropna().quantile(0.95),
        p99=lambda x: x.dropna().quantile(0.99)
    ).reset_index()
    # normalize column names
    agg = agg.rename(columns={component_col: 'component', kpi_col: 'kpi_name'})
    return agg

# Aggregate each file (store full aggregated tables in variables)
metric_aggr = aggregate_kpi(metric, 'cmdb_id', 'kpi_name')
trace_aggr = aggregate_kpi(trace, 'cmdb_id', 'trace_name')
log_aggr = aggregate_kpi(log, 'cmdb_id', 'log_name')

# Prepare compact summaries: top 20 rows by count with requested columns
metric_summary = metric_aggr.sort_values('count', ascending=False).head(20)[['component','kpi_name','count','p50','p95','p99']]
trace_summary = trace_aggr.sort_values('count', ascending=False).head(20)[['component','kpi_name','count','p50','p95','p99']]
log_summary = log_aggr.sort_values('count', ascending=False).head(20)[['component','kpi_name','count','p50','p95','p99']]

# Error logs summary: unique cmdb_id values and total number of error log entries
error_log_cmdbs = error_logs['cmdb_id'].dropna().unique()
error_log_count = len(error_logs)

# Display compact results (use variable names as required)
metric_summary, trace_summary, log_summary, error_log_cmdbs, error_log_count

```
Out[1]:
```
metric_summary, trace_summary, log_summary, error_log_cmdbs, error_log_count  # reuse existing vars to confirm availability

summary = (
    "Summary of results:\n"
    "- Aggregations were computed for metric.csv, trace.csv, and log.csv by (cmdb_id, kpi).\n"
    "- Top summaries (showing up to 20 rows each) were produced and stored in variables:\n"
    "  metric_summary, trace_summary, log_summary. Full aggregated tables are in metric_aggr, trace_aggr, log_aggr.\n"
    "- Observations from the top summaries:\n"
    "  * Many top metric and trace KPI groups have count = 31 data points (seen across the top rows).\n"
    "  * metric top rows include JVM/OS and Tomcat-related KPIs (e.g., IG01/IG02 JVM memory, many Tomcat04 KPIs).\n"
    "  * trace top rows show dockerA/dockerB related traces (duration and row_count series) with p50/p95/p99 values computed.\n"
    "  * log top rows include Mysql01, Tomcat01–04 and apache01/02 log.row_count and log.error_count series. Several log.error_count groups have p50/p95/p99 = 0.\n"
    f"- error_logs.csv contains {error_log_count} total entries and unique cmdb_id values: {list(error_log_cmdbs)} (empty means no error log records found).\n"
    "\n"
    "If you want, I can now filter these aggregated series by a specific time range or compute anomaly thresholds using the global series."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     component                                           kpi_name  count           p50           p95           p99
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09
1569  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00
1573  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00
88        IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31  2.040136e+09  2.040136e+09  2.040136e+09
90        IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31  4.186135e+08  7.895955e+08  8.247816e+08
91        IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.853378e+07  5.853378e+07  5.853378e+07
94        IG02           JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.100000e+01  7.500000e+01  7.640000e+01
1564  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  1.297644e+09  1.301344e+09  1.301596e+09
1565  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  2.181426e+06  2.187160e+06  2.187716e+06
1566  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00
1579  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  4.000000e+00  1.200000e+01  3.070000e+01
1567  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00  0.000000e+00  0.000000e+00
1556  Tomcat04                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  7.774386e+06  7.775198e+06  7.775270e+06
1545  Tomcat04      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31  1.000000e+00  1.000000e+00  1.000000e+00
1548  Tomcat04  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  4.000000e-01  4.000000e-01  4.000000e-01
1570  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00
1571  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00  0.000000e+00  0.000000e+00
1572  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00
1574  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00
1575  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00,     component                       kpi_name  count          p50          p95          p99
260  dockerB2        trace.to_MG02.row_count     31    72.000000   122.500000   126.500000
259  dockerB2     trace.to_MG02.duration_p95     31     0.696750    14.950350    35.889140
258  dockerB2    trace.to_MG02.duration_mean     31     0.071400     2.703143     6.363153
225  dockerB1  trace.from_MG01.duration_mean     31     0.353536     2.471035     3.614222
226  dockerB1   trace.from_MG01.duration_p95     31     2.096800    13.709750    21.821580
211  dockerA2   trace.from_MG02.duration_p95     31     1.028000     9.765425    32.674955
212  dockerA2      trace.from_MG02.row_count     31   156.000000   208.000000   220.500000
216  dockerA2       trace.self.duration_mean     31     0.019634     0.172235     0.694573
217  dockerA2        trace.self.duration_p95     31     0.007000     0.016550     0.712830
218  dockerA2           trace.self.row_count     31  2244.000000  2995.500000  3041.300000
207  dockerA2  trace.from_MG01.duration_mean     31     0.151203     1.176020     5.410089
208  dockerA2   trace.from_MG01.duration_p95     31     1.028000    10.053400    33.923695
209  dockerA2      trace.from_MG01.row_count     31   166.000000   210.500000   212.400000
210  dockerA2  trace.from_MG02.duration_mean     31     0.148876     1.204172     4.417088
195  dockerA1  trace.from_MG02.duration_mean     31     0.144099     1.200819     3.235365
196  dockerA1   trace.from_MG02.duration_p95     31     1.030000    10.495950    27.095785
197  dockerA1      trace.from_MG02.row_count     31   157.000000   218.500000   221.700000
198  dockerA1       trace.self.duration_mean     31     0.018672     0.175078     0.565344
199  dockerA1        trace.self.duration_p95     31     0.007000     0.197375     0.799600
200  dockerA1           trace.self.row_count     31  2213.000000  3051.000000  3085.600000,    component          kpi_name  count      p50       p95       p99
0    Mysql01   log.error_count     31    0.000    0.0000    0.0000
1    Mysql01     log.row_count     31    7.000   22.0000   38.4000
6   Tomcat01     log.row_count     31  429.000  472.5000  482.3000
4   Tomcat01   log.error_count     31    0.000    0.0000    0.0000
18  apache02   log.error_count     31    0.000    0.0000    0.0000
17  apache01     log.row_count     31  853.000  943.0000  969.2000
7   Tomcat02   log.error_count     31    0.000    0.0000    0.0000
9   Tomcat02     log.row_count     31  429.000  500.0000  515.3000
10  Tomcat03   log.error_count     31    0.000    0.0000    0.0000
15  Tomcat04     log.row_count     31  432.000  491.0000  503.4000
13  Tomcat04   log.error_count     31    0.000    0.0000    0.0000
12  Tomcat03     log.row_count     31  435.000  471.5000  476.5000
19  apache02     log.row_count     31  863.000  937.5000  958.2000
16  apache01   log.error_count     31    0.000    0.0000    0.0000
5   Tomcat01  log.gc_pause_max     22    0.080    0.1690    0.1858
14  Tomcat04  log.gc_pause_max     22    0.075    0.1785    0.4091
11  Tomcat03  log.gc_pause_max     21    0.070    0.1100    0.1820
8   Tomcat02  log.gc_pause_max     21    0.070    0.1100    0.1100
3    Mysql02     log.row_count      3    1.000    1.0000    1.0000
2    Mysql02   log.error_count      3    0.000    0.0000    0.0000, array([], dtype=object), 0)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing variables: metric, trace, log, metric_aggr, trace_aggr, log_aggr
# Ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# Define window (inclusive) in UTC+8
start = tz.localize(pd.Timestamp("2021-03-24 16:30:00"))
end = tz.localize(pd.Timestamp("2021-03-24 17:00:00"))

# Helper to ensure timestamp column is tz-aware in UTC+8
def ensure_ts_tz(df):
    df = df.copy()
    if 'timestamp' in df.columns:
        # If dtype is datetime and tz-aware, convert; if naive, localize from UTC then convert
        if pd.api.types.is_datetime64_any_dtype(df['timestamp']):
            if df['timestamp'].dt.tz is None:
                # assume originally in UTC -> convert to UTC then to Asia/Shanghai
                df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert(tz)
            else:
                df['timestamp'] = df['timestamp'].dt.tz_convert(tz)
        else:
            # assume integer seconds -> convert from UTC then to Asia/Shanghai
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    return df

metric = ensure_ts_tz(metric)
trace = ensure_ts_tz(trace)
log = ensure_ts_tz(log)

# Generic processor
def detect_anomalies_in_window(raw_df, raw_kpi_col, aggr_df):
    # raw_kpi_col: column name in raw df for KPI (e.g., 'kpi_name', 'trace_name', 'log_name')
    df = raw_df.copy()
    df['kpi_name'] = df[raw_kpi_col]
    # Filter window inclusive
    df_window = df[(df['timestamp'] >= start) & (df['timestamp'] <= end)].copy()
    if df_window.shape[0] == 0:
        # Return empty structures
        empty_anoms = pd.DataFrame(columns=df.columns.tolist() + ['p50','p95','anomaly_type'])
        full_group = pd.DataFrame(columns=['component','kpi_name','anomaly_type','anomaly_count_in_window',
                                           'earliest_anomaly_timestamp','max_value_in_window','p50','p95'])
        return df_window, empty_anoms, full_group

    # Merge thresholds from aggr (aggr has columns: component, kpi_name, p50, p95)
    merged = df_window.merge(aggr_df[['component','kpi_name','p50','p95']],
                             left_on=['cmdb_id','kpi_name'],
                             right_on=['component','kpi_name'],
                             how='left',
                             suffixes=('','_aggr'))
    # Ensure numeric value
    merged['value'] = pd.to_numeric(merged['value'], errors='coerce')
    # Drop rows without thresholds (p95 NaN) since can't evaluate
    merged = merged.dropna(subset=['p95','p50'])
    if merged.shape[0] == 0:
        empty_anoms = pd.DataFrame(columns=merged.columns.tolist() + ['anomaly_type'])
        full_group = pd.DataFrame(columns=['component','kpi_name','anomaly_type','anomaly_count_in_window',
                                           'earliest_anomaly_timestamp','max_value_in_window','p50','p95'])
        return df_window, empty_anoms, full_group

    # High-sided anomaly
    merged_high = merged[merged['value'] >= merged['p95']].copy()
    merged_high['anomaly_type'] = 'high'

    # Low-sided anomaly for kpis containing 'row_count' (case-sensitive per instruction)
    is_row_count = merged['kpi_name'].str.contains('row_count', na=False)
    merged_low = merged[is_row_count & (merged['value'] <= merged['p50'] * 0.5)].copy()
    merged_low['anomaly_type'] = 'low'

    # Combine anomalies (a row could appear in both; keep both records if so)
    anoms = pd.concat([merged_high, merged_low], ignore_index=True, sort=False)
    # If empty
    if anoms.shape[0] == 0:
        full_group = pd.DataFrame(columns=['component','kpi_name','anomaly_type','anomaly_count_in_window',
                                           'earliest_anomaly_timestamp','max_value_in_window','p50','p95'])
        return df_window, anoms, full_group

    # Group as requested by (component, kpi_name, anomaly_type)
    grp = (anoms
           .groupby(['cmdb_id','kpi_name','anomaly_type'], dropna=False)
           .agg(anomaly_count_in_window=('value','count'),
                earliest_anomaly_timestamp=('timestamp','min'),
                max_value_in_window=('value','max'),
                p50=('p50','first'),
                p95=('p95','first'))
           .reset_index()
          )
    # Rename cmdb_id -> component
    grp = grp.rename(columns={'cmdb_id':'component'})
    # Ensure earliest timestamp is tz-aware (it should be)
    # Sort descending by anomaly_count_in_window
    grp = grp.sort_values('anomaly_count_in_window', ascending=False)
    return df_window, anoms, grp

# Process each file
metric_window, metric_anoms_window, metric_group = detect_anomalies_in_window(metric, 'kpi_name', metric_aggr)
trace_window, trace_anoms_window, trace_group = detect_anomalies_in_window(trace, 'trace_name', trace_aggr)
log_window, log_anoms_window, log_group = detect_anomalies_in_window(log, 'log_name', log_aggr)

# Combine grouped summaries across files
combined_group = pd.concat([metric_group, trace_group, log_group], ignore_index=True, sort=False)
# If empty, keep consistent columns
if combined_group.shape[0] == 0:
    combined_group = combined_group[['component','kpi_name','anomaly_type','anomaly_count_in_window',
                                     'earliest_anomaly_timestamp','max_value_in_window','p50','p95']]

# Format earliest_anomaly_timestamp as tz-aware strings for compact display
if 'earliest_anomaly_timestamp' in combined_group.columns and combined_group.shape[0] > 0:
    combined_group['earliest_anomaly_timestamp'] = combined_group['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Top 20 rows by anomaly_count_in_window
grouped_summary_top20 = combined_group.sort_values('anomaly_count_in_window', ascending=False).head(20)

# Counts of total anomaly rows detected per file (full anomaly records within window)
metric_count = int(metric_anoms_window.shape[0])
trace_count = int(trace_anoms_window.shape[0])
log_count = int(log_anoms_window.shape[0])

# Store full anomaly rows in kernel variables as requested (already stored: metric_anoms_window, trace_anoms_window, log_anoms_window)
# Final compact outputs: grouped_summary_top20 and counts tuple
grouped_summary_top20, (metric_count, trace_count, log_count)

```
Out[2]:
```
summary = (
    "Summary for window 2021-03-24 16:30 to 17:00 (UTC+8):\n"
    "- Total anomaly rows detected in the window: metric=16,423, trace=633, log=255.\n"
    "- Top grouped anomalies (most groups have 31 anomaly points each) include:\n"
    "  * Tomcat components (Tomcat01, Tomcat02, Tomcat04) — many Session- and Request-related KPIs flagged as 'high'.\n"
    "  * IG01 JVM memory KPIs (HeapMemoryMax and NoHeapMemoryUsed) flagged as 'high'.\n"
    "  * MG02 OSLinux_PROCESS_PROCESS_PROCNoZombies flagged as 'high'.\n"
    "  * ServiceTest5 app.rr flagged as 'high'.\n"
    "- Notable data characteristics: several top KPIs have p50 and p95 equal to 0, and were therefore marked 'high' for all points (31 occurrences). Some JVM/OS KPIs have large numeric p95 values and all window values equal those p95 thresholds, producing 'high' flags as well.\n"
    "- Suggested next steps: review KPI definitions and thresholds for series with p95==0 (to avoid spurious high flags), and inspect the JVM/OS groups (IG01, MG02) and Tomcat session/request KPIs for underlying issues.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       component                                           kpi_name anomaly_type  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window           p50           p95
63      Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
7       Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
6       Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...         high                       31   2021-03-24 16:30:00+0800         5.000000e+02  5.000000e+02  5.000000e+02
5       Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
4       Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
3       Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
2       Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
1           IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax         high                       31   2021-03-24 16:30:00+0800         2.040136e+09  2.040136e+09  2.040136e+09
0           IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed         high                       31   2021-03-24 16:30:00+0800         5.848308e+07  5.848308e+07  5.848308e+07
15          MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies         high                       31   2021-03-24 16:30:00+0800         1.000000e+00  1.000000e+00  1.000000e+00
14      Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
13      Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
12      Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
11      Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
10      Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...         high                       31   2021-03-24 16:30:00+0800         3.978080e+05  3.978080e+05  3.978080e+05
9       Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
8       Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...         high                       31   2021-03-24 16:30:00+0800         1.051000e+03  1.051000e+03  1.051000e+03
62      Tomcat02        Tomcat-Sessions_7441--_SESSIONActiveCounter         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
61      Tomcat02     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter         high                       31   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00
60  ServiceTest5                                             app.rr         high                       31   2021-03-24 16:30:00+0800         1.000000e+02  1.000000e+02  1.000000e+02, (16423, 633, 255))```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse existing grouped DataFrames from previous steps: metric_group, trace_group, log_group
# Combine full grouped anomaly table
all_groups = pd.concat([metric_group, trace_group, log_group], ignore_index=True, sort=False)

# Ensure numeric columns
all_groups['p95'] = pd.to_numeric(all_groups['p95'], errors='coerce')
all_groups['p50'] = pd.to_numeric(all_groups['p50'], errors='coerce')
all_groups['max_value_in_window'] = pd.to_numeric(all_groups['max_value_in_window'], errors='coerce')
all_groups['anomaly_count_in_window'] = pd.to_numeric(all_groups['anomaly_count_in_window'], errors='coerce')

# Keep only high-sided anomalies
high_groups = all_groups[all_groups['anomaly_type'] == 'high'].copy()

# Compute severity_pct where p95 > 0
high_groups = high_groups[high_groups['p95'] > 0].copy()
high_groups['severity_pct'] = (high_groups['max_value_in_window'] - high_groups['p95']) / high_groups['p95']

# Keep groups with severity_pct > 0.5
filtered_anoms_severe = high_groups[high_groups['severity_pct'] > 0.5].copy()

# Prepare display columns and formatting
# Ensure earliest_anomaly_timestamp is datetime tz-aware; convert to string in UTC+8
tz = pytz.timezone('Asia/Shanghai')
if 'earliest_anomaly_timestamp' in filtered_anoms_severe.columns and not filtered_anoms_severe['earliest_anomaly_timestamp'].empty:
    # If the column is dtype object/string (maybe previously formatted), try to coerce back then format
    try:
        filtered_anoms_severe['earliest_anomaly_timestamp'] = pd.to_datetime(filtered_anoms_severe['earliest_anomaly_timestamp'], utc=True).dt.tz_convert(tz)
    except Exception:
        # If already tz-aware datetimes, keep as is
        pass
    # Format as readable string with offset
    filtered_anoms_severe['earliest_anomaly_timestamp_str'] = filtered_anoms_severe['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Select and sort top results
display_cols = ['component','kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp_str',
                'max_value_in_window','p50','p95','severity_pct']
top20_severe = (filtered_anoms_severe
                .sort_values(['severity_pct','anomaly_count_in_window'], ascending=[False, False])
                .head(20)
               )

# Compact per-component summary: count of severe groups per component
per_component = (filtered_anoms_severe
                 .groupby('component', dropna=False)
                 .agg(number_of_severe_groups=('kpi_name','nunique'))
                 .reset_index()
                 .sort_values('number_of_severe_groups', ascending=False)
                )

# Prepare final compact tables for display: select columns and ensure compactness
# Use the formatted timestamp column if present, otherwise use earliest_anomaly_timestamp directly
if 'earliest_anomaly_timestamp_str' in top20_severe.columns:
    compact_top20 = top20_severe[display_cols].rename(columns={'earliest_anomaly_timestamp_str':'earliest_anomaly_timestamp'})
else:
    compact_top20 = top20_severe[['component','kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp',
                                  'max_value_in_window','p50','p95','severity_pct']]

# Round numeric columns for compact display
for col in ['max_value_in_window','p50','p95','severity_pct']:
    if col in compact_top20.columns:
        compact_top20[col] = compact_top20[col].round(6)

per_component = per_component.head(100)  # keep compact; will be sorted by count desc

# Final outputs: store filtered_anoms_severe in kernel and display compact tables
filtered_anoms_severe, compact_top20.reset_index(drop=True), per_component.reset_index(drop=True)

```
Out[3]:
```
```python
# Summarize severe anomalies found (compact plain-English summary)
# Reuse kernel variables: filtered_anoms_severe, per_component

# Compute key stats
total_severe_groups = int(filtered_anoms_severe.shape[0]) if 'filtered_anoms_severe' in globals() else 0
num_affected_components = int(per_component.shape[0]) if 'per_component' in globals() else 0

# Top components by number_of_severe_groups (safe access)
if 'per_component' in globals() and not per_component.empty:
    top_components_list = list(per_component.head(5).itertuples(index=False, name=None))
else:
    top_components_list = []

# Top single group by severity_pct
if 'filtered_anoms_severe' in globals() and not filtered_anoms_severe.empty:
    top_row = filtered_anoms_severe.sort_values('severity_pct', ascending=False).iloc[0]
    top_group_example = {
        'component': str(top_row['component']),
        'kpi_name': str(top_row['kpi_name']),
        'severity_pct': float(top_row['severity_pct']),
        'max_value_in_window': float(top_row['max_value_in_window']),
        'p95': float(top_row['p95']),
        'earliest_anomaly_timestamp': pd.to_datetime(top_row['earliest_anomaly_timestamp']).tz_convert(pytz.timezone('Asia/Shanghai')).strftime('%Y-%m-%d %H:%M:%S%z')
    }
    # time range of severe anomalies
    try:
        tmin = pd.to_datetime(filtered_anoms_severe['earliest_anomaly_timestamp']).min().tz_convert(pytz.timezone('Asia/Shanghai')).strftime('%Y-%m-%d %H:%M:%S%z')
        tmax = pd.to_datetime(filtered_anoms_severe['earliest_anomaly_timestamp']).max().tz_convert(pytz.timezone('Asia/Shanghai')).strftime('%Y-%m-%d %H:%M:%S%z')
    except Exception:
        tmin = tmax = "N/A"
else:
    top_group_example = {}
    tmin = tmax = "N/A"

summary = (
    f"Severe anomaly summary for 2021-03-24 16:30–17:00 (UTC+8):\n"
    f"- Total severe groups detected: {total_severe_groups}\n"
    f"- Number of affected components: {num_affected_components}\n"
    f"- Top components by number_of_severe_groups (top 5): {top_components_list}\n"
    f"- Example most severe group:\n"
    f"    component = {top_group_example.get('component','N/A')},\n"
    f"    kpi_name = {top_group_example.get('kpi_name','N/A')},\n"
    f"    severity_pct = {top_group_example.get('severity_pct','N/A'):.6f},\n"
    f"    max_value_in_window = {top_group_example.get('max_value_in_window','N/A')}, p95 = {top_group_example.get('p95','N/A')},\n"
    f"    earliest_anomaly_timestamp = {top_group_example.get('earliest_anomaly_timestamp','N/A')}\n"
    f"- Severe-anomaly timestamps span: {tmin}  to  {tmax}\n\n"
    "Interpretation / next steps:\n"
    "- Many severe groups are concentrated in MG01/MG02 and Tomcat nodes (see top components above).\n"
    "- The highest-severity examples are trace duration and application-level KPIs where max values exceed p95 by multiple times.\n"
    "- Investigate the top components (MG01, MG02, Tomcat04, Tomcat03, Tomcat02) and the listed KPIs for root causes (e.g., service-side latency, resource contention).\n"
    "- If you want, I can show the top 20 severe groups table and the per-component counts in detail."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         component                         kpi_name anomaly_type  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window       p50        p95  severity_pct earliest_anomaly_timestamp_str
1097      Tomcat01           OSLinux-CPU_CPU_CPUWio         high                        2  2021-03-24 16:31:00+08:00             0.956023  0.025000   0.563313      0.697142       2021-03-24 16:31:00+0800
1100  ServiceTest9                          app.mrt         high                        2  2021-03-24 16:31:00+08:00            23.164030  0.333070   6.348970      2.648471       2021-03-24 16:31:00+0800
1108  ServiceTest3                          app.mrt         high                        2  2021-03-24 16:31:00+08:00            26.776220  0.314360   5.073360      4.277808       2021-03-24 16:31:00+0800
1110  ServiceTest4                          app.mrt         high                        2  2021-03-24 16:31:00+08:00             9.591350  1.936650   6.281445      0.526934       2021-03-24 16:31:00+0800
1115  ServiceTest6                          app.mrt         high                        2  2021-03-24 16:31:00+08:00             5.768490  0.645370   3.808240      0.514739       2021-03-24 16:31:00+0800
...            ...                              ...          ...                      ...                        ...                  ...       ...        ...           ...                            ...
2003          IG01   trace.to_Tomcat03.duration_p95         high                        2  2021-03-24 16:31:00+08:00            13.745250  2.042100   8.382250      0.639804       2021-03-24 16:31:00+0800
2005          IG01  trace.to_Tomcat04.duration_mean         high                        2  2021-03-24 16:31:00+08:00             3.767089  0.410668   1.718457      1.192135       2021-03-24 16:31:00+0800
2112       Mysql01                    log.row_count         high                        2  2021-03-24 16:30:00+08:00            45.000000  7.000000  22.000000      1.045455       2021-03-24 16:30:00+0800
2117      Tomcat04                 log.gc_pause_max         high                        2  2021-03-24 16:32:00+08:00             0.470000  0.075000   0.178500      1.633053       2021-03-24 16:32:00+0800
2119      Tomcat03                 log.gc_pause_max         high                        2  2021-03-24 16:31:00+08:00             0.200000  0.070000   0.110000      0.818182       2021-03-24 16:31:00+0800

[195 rows x 10 columns],        component                                           kpi_name  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window       p50        p95  severity_pct
0       dockerA2                            trace.self.duration_p95                        2   2021-03-24 16:30:00+0800             1.008000  0.007000   0.016550     59.906344
1           MG01                    trace.to_dockerA2.duration_mean                        2   2021-03-24 16:31:00+0800             6.964884  0.151203   1.176020      4.922420
2       dockerA2                      trace.from_MG01.duration_mean                        2   2021-03-24 16:31:00+0800             6.964884  0.151203   1.176020      4.922420
3        Mysql01                    Mysql-MySQL_3306_ThreadsRunning                        2   2021-03-24 16:39:00+0800             9.000000  1.000000   1.550000      4.806452
4   ServiceTest3                                            app.mrt                        2   2021-03-24 16:31:00+0800            26.776220  0.314360   5.073360      4.277808
5       dockerA1                            trace.self.duration_p95                        2   2021-03-24 16:31:00+0800             1.012000  0.007000   0.197375      4.127296
6       dockerA2                           trace.self.duration_mean                        2   2021-03-24 16:31:00+0800             0.874767  0.019634   0.172235      4.078915
7           MG01                    trace.to_dockerA1.duration_mean                        2   2021-03-24 16:31:00+0800             6.229045  0.145472   1.229430      4.066614
8       dockerA1                      trace.from_MG01.duration_mean                        2   2021-03-24 16:31:00+0800             6.229045  0.145472   1.229430      4.066614
9       dockerA2                      trace.from_MG02.duration_mean                        2   2021-03-24 16:31:00+0800             5.478490  0.148876   1.204172      3.549593
10          MG02                    trace.to_dockerA2.duration_mean                        2   2021-03-24 16:31:00+0800             5.478490  0.148876   1.204172      3.549593
11      dockerA2                        trace.to_MG02.duration_mean                        2   2021-03-24 16:31:00+0800            24.687865  0.063016   5.899713      3.184587
12          MG02                  trace.from_dockerA2.duration_mean                        2   2021-03-24 16:31:00+0800            24.687865  0.063016   5.899713      3.184587
13      dockerA2                       trace.from_MG01.duration_p95                        2   2021-03-24 16:31:00+0800            41.440150  1.028000  10.053400      3.122004
14          MG01                     trace.to_dockerA2.duration_p95                        2   2021-03-24 16:31:00+0800            41.440150  1.028000  10.053400      3.122004
15      dockerA2                       trace.from_MG02.duration_p95                        2   2021-03-24 16:31:00+0800            39.642650  1.028000   9.765425      3.059490
16          MG02                     trace.to_dockerA2.duration_p95                        2   2021-03-24 16:31:00+0800            39.642650  1.028000   9.765425      3.059490
17      Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                        1   2021-03-24 16:46:00+0800             0.200000  0.000000   0.050000      3.000000
18       Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                        1   2021-03-24 16:54:00+0800             0.400000  0.000000   0.100000      3.000000
19      Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                        1   2021-03-24 17:00:00+0800             2.000000  0.000000   0.500000      3.000000,        component  number_of_severe_groups
0           MG02                       30
1           MG01                       28
2       Tomcat04                       17
3       Tomcat03                       15
4       Tomcat02                       14
..           ...                      ...
17  ServiceTest7                        1
18  ServiceTest3                        1
19  ServiceTest9                        1
20  ServiceTest4                        1
21  ServiceTest6                        1

[22 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse kernel variables: trace_anoms_window, trace_aggr
tz = pytz.timezone('Asia/Shanghai')

# Safety check: ensure trace_anoms_window exists
if 'trace_anoms_window' not in globals():
    trace_severe_details = pd.DataFrame()  # empty
    compact_top20 = pd.DataFrame()
    unique_components_count = 0
    top10_trace_names = pd.DataFrame()
else:
    df = trace_anoms_window.copy()
    # Ensure required columns exist: cmdb_id, trace_name, value, timestamp, anomaly_type
    # Some rows may have 'trace_name' under 'kpi_name' depending on earlier merges; handle both
    if 'trace_name' not in df.columns and 'kpi_name' in df.columns:
        df['trace_name'] = df['kpi_name']
    # Ensure timestamp is tz-aware in Asia/Shanghai
    if 'timestamp' in df.columns:
        try:
            # if already datetime with tz, convert; else parse as UTC then convert
            if pd.api.types.is_datetime64_any_dtype(df['timestamp']):
                if df['timestamp'].dt.tz is None:
                    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert(tz)
                else:
                    df['timestamp'] = df['timestamp'].dt.tz_convert(tz)
            else:
                df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        except Exception:
            df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert(tz)

    # Group anomaly rows by component+trace_name+anomaly_type to produce the requested aggregated fields
    group_cols = []
    if 'cmdb_id' in df.columns:
        group_cols.append('cmdb_id')
    elif 'component' in df.columns:
        df = df.rename(columns={'component':'cmdb_id'})
        group_cols.append('cmdb_id')
    else:
        # no component column -> create placeholder and continue
        df['cmdb_id'] = None
        group_cols.append('cmdb_id')

    group_cols += ['trace_name','anomaly_type']

    grouped = (df
               .groupby(group_cols, dropna=False)
               .agg(anomaly_count_in_window=('value','count'),
                    earliest_anomaly_timestamp=('timestamp','min'),
                    max_value_in_window=('value','max'))
               .reset_index()
              )

    # Join thresholds from trace_aggr (which uses columns 'component' and 'kpi_name')
    # Prepare trace_aggr keys
    ta = trace_aggr.copy()
    # Ensure ta has 'component' and 'kpi_name'
    if 'component' not in ta.columns and 'cmdb_id' in ta.columns:
        ta = ta.rename(columns={'cmdb_id':'component'})
    # Merge
    merged = grouped.merge(ta[['component','kpi_name','p50','p95']],
                           left_on=['cmdb_id','trace_name'],
                           right_on=['component','kpi_name'],
                           how='left')
    # Filter for high anomalies only
    merged_high = merged[merged['anomaly_type'] == 'high'].copy()

    # Ensure numeric fields
    for col in ['p95','p50','max_value_in_window']:
        merged_high[col] = pd.to_numeric(merged_high[col], errors='coerce')

    # Compute severity_pct where p95 > 0
    merged_high = merged_high[merged_high['p95'] > 0].copy()
    merged_high['severity_pct'] = (merged_high['max_value_in_window'] - merged_high['p95']) / merged_high['p95']

    # Keep only severity_pct > 0.5
    trace_severe_details = merged_high[merged_high['severity_pct'] > 0.5].copy()

    # Prepare display formatting
    if 'earliest_anomaly_timestamp' in trace_severe_details.columns:
        try:
            trace_severe_details['earliest_anomaly_timestamp'] = pd.to_datetime(trace_severe_details['earliest_anomaly_timestamp']).dt.tz_convert(tz)
        except Exception:
            trace_severe_details['earliest_anomaly_timestamp'] = pd.to_datetime(trace_severe_details['earliest_anomaly_timestamp'], utc=True).dt.tz_convert(tz)
        trace_severe_details['earliest_anomaly_timestamp_str'] = trace_severe_details['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

    # Select and sort top 20 by severity_pct desc then anomaly_count_in_window desc
    display_cols = ['cmdb_id','trace_name','anomaly_count_in_window','earliest_anomaly_timestamp_str',
                    'max_value_in_window','p50','p95','severity_pct']
    compact_top20 = (trace_severe_details
                     .sort_values(['severity_pct','anomaly_count_in_window'], ascending=[False, False])
                     .head(20)
                     [display_cols]
                     .rename(columns={'cmdb_id':'component','earliest_anomaly_timestamp_str':'earliest_anomaly_timestamp'})
                    )

    # Round numeric columns for compact display
    for col in ['max_value_in_window','p50','p95','severity_pct']:
        if col in compact_top20.columns:
            compact_top20[col] = compact_top20[col].round(6)

    # (a) count of unique components among filtered trace severe records
    unique_components_count = int(trace_severe_details['cmdb_id'].nunique(dropna=True))

    # (b) top 10 distinct trace_name patterns with counts
    top10_trace_names = (trace_severe_details
                         .groupby('trace_name', dropna=False)
                         .agg(count=('trace_name','size'))
                         .reset_index()
                         .sort_values('count', ascending=False)
                         .head(10)
                        )

# Final outputs (compact): compact_top20, unique_components_count, top10_trace_names
compact_top20, unique_components_count, top10_trace_names

```
Out[4]:
```
```python
# Compose a plain-English summary using existing kernel variables from the previous steps:
# compact_top20, unique_components_count, top10_trace_names, trace_severe_details

# Build summary
top_examples = []
if 'compact_top20' in globals() and not compact_top20.empty:
    for _, r in compact_top20.head(3).iterrows():
        top_examples.append(f"{r['component']} / {r['trace_name']} (severity_pct={r['severity_pct']})")

top_trace_patterns = []
if 'top10_trace_names' in globals() and not top10_trace_names.empty:
    top_trace_patterns = [f"{row['trace_name']} ({int(row['count'])})" for _, row in top10_trace_names.iterrows()]

summary = (
    "Trace severe-anomaly summary (severity_pct > 0.5, high-sided) for 2021-03-24 16:30–17:00 (UTC+8):\n"
    f"- Number of unique components affected: {unique_components_count}\n"
    f"- Top severe records (examples): {', '.join(top_examples) if top_examples else 'None'}\n"
    f"- Top trace name patterns (top 10): {', '.join(top_trace_patterns) if top_trace_patterns else 'None'}\n"
    "\nKey observations:\n"
    "- The highest-severity trace is trace.self.duration_p95 on dockerA2 (severity_pct ≈ 59.9x), indicating a very large spike vs historical p95.\n"
    "- Multiple high-severity duration traces involve MG01/MG02 and dockerA1/dockerA2 (trace.to_*/trace.from_* and trace.self.duration_mean/p95), with severity_pct commonly >3x–4x.\n"
    "- Tomcat04 also appears among affected components for trace-to/from durations.\n"
    "\nRecommended next steps:\n"
    "- Prioritize investigation of dockerA2 and MG01/MG02 service paths (high latency traces).\n"
    "- Inspect the specific trace types listed above (trace.self.duration_*, trace.to_*/trace.from_* duration metrics) to find root causes (resource contention, downstream slowness, or network issues).\n"
    "- Correlate these trace spikes with service logs and host metrics for the same timestamps to narrow root cause."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    component                         trace_name  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window       p50        p95  severity_pct
282  dockerA2            trace.self.duration_p95                        2   2021-03-24 16:30:00+0800             1.008000  0.007000   0.016550     59.906344
89       MG01    trace.to_dockerA2.duration_mean                        2   2021-03-24 16:31:00+0800             6.964884  0.151203   1.176020      4.922420
270  dockerA2      trace.from_MG01.duration_mean                        2   2021-03-24 16:31:00+0800             6.964884  0.151203   1.176020      4.922420
259  dockerA1            trace.self.duration_p95                        2   2021-03-24 16:31:00+0800             1.012000  0.007000   0.197375      4.127296
281  dockerA2           trace.self.duration_mean                        2   2021-03-24 16:31:00+0800             0.874767  0.019634   0.172235      4.078915
85       MG01    trace.to_dockerA1.duration_mean                        2   2021-03-24 16:31:00+0800             6.229045  0.145472   1.229430      4.066614
250  dockerA1      trace.from_MG01.duration_mean                        2   2021-03-24 16:31:00+0800             6.229045  0.145472   1.229430      4.066614
144      MG02    trace.to_dockerA2.duration_mean                        2   2021-03-24 16:31:00+0800             5.478490  0.148876   1.204172      3.549593
274  dockerA2      trace.from_MG02.duration_mean                        2   2021-03-24 16:31:00+0800             5.478490  0.148876   1.204172      3.549593
121      MG02  trace.from_dockerA2.duration_mean                        2   2021-03-24 16:31:00+0800            24.687865  0.063016   5.899713      3.184587
289  dockerA2        trace.to_MG02.duration_mean                        2   2021-03-24 16:31:00+0800            24.687865  0.063016   5.899713      3.184587
90       MG01     trace.to_dockerA2.duration_p95                        2   2021-03-24 16:31:00+0800            41.440150  1.028000  10.053400      3.122004
271  dockerA2       trace.from_MG01.duration_p95                        2   2021-03-24 16:31:00+0800            41.440150  1.028000  10.053400      3.122004
145      MG02     trace.to_dockerA2.duration_p95                        2   2021-03-24 16:31:00+0800            39.642650  1.028000   9.765425      3.059490
275  dockerA2       trace.from_MG02.duration_p95                        2   2021-03-24 16:31:00+0800            39.642650  1.028000   9.765425      3.059490
258  dockerA1           trace.self.duration_mean                        2   2021-03-24 16:31:00+0800             0.696337  0.018672   0.175078      2.977284
66       MG01  trace.from_dockerA2.duration_mean                        2   2021-03-24 16:31:00+0800            23.187632  0.064470   6.060918      2.825763
285  dockerA2        trace.to_MG01.duration_mean                        2   2021-03-24 16:31:00+0800            23.187632  0.064470   6.060918      2.825763
58       MG01  trace.from_Tomcat04.duration_mean                        2   2021-03-24 16:31:00+0800             6.829595  0.268431   1.846428      2.698814
242  Tomcat04        trace.to_MG01.duration_mean                        2   2021-03-24 16:31:00+0800             6.829595  0.268431   1.846428      2.698814, 12,                        trace_name  count
24       trace.self.duration_mean     11
28    trace.to_MG02.duration_mean      8
26    trace.to_MG01.duration_mean      8
27     trace.to_MG01.duration_p95      8
29     trace.to_MG02.duration_p95      8
25        trace.self.duration_p95      6
0   trace.from_IG01.duration_mean      4
7    trace.from_MG02.duration_p95      4
6   trace.from_MG02.duration_mean      4
5    trace.from_MG01.duration_p95      4)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse kernel variables: metric_anoms_window, metric_aggr
tz = pytz.timezone('Asia/Shanghai')

# Safety: ensure metric_anoms_window exists
if 'metric_anoms_window' not in globals():
    mg_metric_anoms = pd.DataFrame()  # empty
    grouped_summary_top20 = pd.DataFrame()
    per_component_counts = pd.DataFrame()
    top10_kpis = pd.DataFrame()
else:
    df = metric_anoms_window.copy()

    # Normalize component column name to 'cmdb_id'
    if 'cmdb_id' not in df.columns and 'component' in df.columns:
        df = df.rename(columns={'component':'cmdb_id'})

    # Ensure trace/kpi column is 'kpi_name' in metrics (should already be)
    if 'kpi_name' not in df.columns and 'kpi' in df.columns:
        df = df.rename(columns={'kpi':'kpi_name'})

    # Ensure timestamp tz-aware in Asia/Shanghai
    if 'timestamp' in df.columns:
        if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            # if datetime but naive, assume UTC then convert; else convert tz
            if df['timestamp'].dt.tz is None:
                df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert(tz)
            else:
                df['timestamp'] = df['timestamp'].dt.tz_convert(tz)

    # Filter to MG01 and MG02
    mg_metric_anoms = df[df['cmdb_id'].isin(['MG01','MG02'])].copy()

    # Join to metric_aggr to get p50/p95 (metric_aggr has 'component' and 'kpi_name')
    ma = metric_aggr.copy()
    if 'component' not in ma.columns and 'cmdb_id' in ma.columns:
        ma = ma.rename(columns={'cmdb_id':'component'})
    ma_sel = ma[['component','kpi_name','p50','p95']].drop_duplicates()

    mg_metric_anoms = mg_metric_anoms.merge(ma_sel,
                                            left_on=['cmdb_id','kpi_name'],
                                            right_on=['component','kpi_name'],
                                            how='left',
                                            suffixes=('','_aggr'))

    # Ensure numeric fields
    mg_metric_anoms['value'] = pd.to_numeric(mg_metric_anoms.get('value', pd.Series(dtype=float)), errors='coerce')
    mg_metric_anoms['p95'] = pd.to_numeric(mg_metric_anoms['p95'], errors='coerce')
    mg_metric_anoms['p50'] = pd.to_numeric(mg_metric_anoms['p50'], errors='coerce')

    # Group by (component, kpi_name)
    grouped = (mg_metric_anoms
               .groupby(['cmdb_id','kpi_name'], dropna=False)
               .agg(anomaly_count_in_window=('value','count'),
                    earliest_anomaly_timestamp=('timestamp','min'),
                    max_value_in_window=('value','max'),
                    p50=('p50','first'),
                    p95=('p95','first'))
               .reset_index()
              )

    # Compute severity_pct where p95 > 0
    grouped['p95'] = pd.to_numeric(grouped['p95'], errors='coerce')
    grouped['max_value_in_window'] = pd.to_numeric(grouped['max_value_in_window'], errors='coerce')
    grouped['severity_pct'] = None
    mask = grouped['p95'] > 0
    grouped.loc[mask, 'severity_pct'] = (grouped.loc[mask, 'max_value_in_window'] - grouped.loc[mask, 'p95']) / grouped.loc[mask, 'p95']

    # Format earliest_anomaly_timestamp for compact display (Asia/Shanghai)
    if 'earliest_anomaly_timestamp' in grouped.columns:
        try:
            grouped['earliest_anomaly_timestamp'] = pd.to_datetime(grouped['earliest_anomaly_timestamp']).dt.tz_convert(tz)
        except Exception:
            grouped['earliest_anomaly_timestamp'] = pd.to_datetime(grouped['earliest_anomaly_timestamp'], utc=True).dt.tz_convert(tz)
        grouped['earliest_anomaly_timestamp_str'] = grouped['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

    # Sort by severity_pct desc (NaNs last) then anomaly_count_in_window desc
    grouped_sorted = grouped.sort_values(by=['severity_pct','anomaly_count_in_window'], ascending=[False, False])

    # Compact top 20 display
    display_cols = ['cmdb_id','kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp_str',
                    'max_value_in_window','p50','p95','severity_pct']
    grouped_summary_top20 = (grouped_sorted.head(20)[display_cols]
                             .rename(columns={'cmdb_id':'component','earliest_anomaly_timestamp_str':'earliest_anomaly_timestamp'}))
    # Round numeric columns for compactness
    for col in ['max_value_in_window','p50','p95','severity_pct']:
        if col in grouped_summary_top20.columns:
            grouped_summary_top20[col] = pd.to_numeric(grouped_summary_top20[col], errors='coerce').round(6)

    # (a) per-component summary showing counts of distinct severe KPIs for MG01 and MG02
    # Define severe as anomaly_type == 'high' AND severity_pct > 0.5
    # Ensure anomaly_type exists in mg_metric_anoms
    if 'anomaly_type' in mg_metric_anoms.columns:
        mg_metric_anoms['anomaly_type'] = mg_metric_anoms['anomaly_type'].astype(str)
    else:
        mg_metric_anoms['anomaly_type'] = None

    # Build grouped severity table per kpi to evaluate severity_pct
    kpi_group = grouped_sorted.copy()
    # Determine severe flag
    kpi_group['is_severe'] = False
    severe_mask = (kpi_group['severity_pct'].notna()) & (kpi_group['severity_pct'] > 0.5)
    # To check anomaly_type == 'high' at kpi level, see if any corresponding rows in mg_metric_anoms for that kpi are high
    # Build a helper mapping of (cmdb_id,kpi_name) -> has_high (boolean)
    high_flags = (mg_metric_anoms[mg_metric_anoms['anomaly_type']=='high']
                  .groupby(['cmdb_id','kpi_name'])
                  .size()
                  .reset_index()[['cmdb_id','kpi_name']])
    high_flags['has_high'] = True
    kpi_group = kpi_group.merge(high_flags, left_on=['cmdb_id','kpi_name'], right_on=['cmdb_id','kpi_name'], how='left')
    kpi_group['has_high'] = kpi_group['has_high'].fillna(False)

    kpi_group.loc[severe_mask & kpi_group['has_high'], 'is_severe'] = True

    per_component_counts = (kpi_group[kpi_group['is_severe']]
                            .groupby('cmdb_id', dropna=False)
                            .agg(number_of_severe_kpis=('kpi_name','nunique'))
                            .reindex(['MG01','MG02'], fill_value=0)  # ensure MG01/MG02 rows present
                            .reset_index()
                            .rename(columns={'cmdb_id':'component'}))

    # (b) top 10 kpi_name patterns with counts among MG01/MG02 metric anomalies (use mg_metric_anoms)
    top10_kpis = (mg_metric_anoms
                  .groupby('kpi_name', dropna=False)
                  .agg(count=('kpi_name','size'))
                  .reset_index()
                  .sort_values('count', ascending=False)
                  .head(10))

# Final outputs (compact): grouped_summary_top20, per_component_counts, top10_kpis
mg_metric_anoms, grouped_summary_top20, per_component_counts, top10_kpis

```
Out[5]:
```
Summary of MG01/MG02 metric anomalies (window 2021-03-24 16:30–17:00, UTC+8):

- Data stored: full filtered metric anomaly rows for MG01/MG02 are saved as mg_metric_anoms (1711 rows).
- Overall: MG01 shows more severe metric KPI breaches than MG02.
  - MG01: 5 distinct KPIs flagged as severe (severity_pct > 0.5 and high-sided).
  - MG02: 2 distinct KPIs flagged as severe.

Key findings (top compact results):
- Highest relative breaches (examples from the top results):
  - MG01 OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps — severity_pct = 1.40 (max = 6.0 vs p95 = 2.5).
  - MG01 OSLinux-CPU_CPU_CPUWio — severity_pct ≈ 1.03 (max = 3.765 vs p95 = 1.858).
  - MG01 OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT — severity_pct = 1.00 (max = 1 vs p95 = 0.5).
  - MG02 notable disk breaches: sda_DSKRead, DSKTps, DSKBps with severity_pct ≈ 0.72–0.875.
  - Several JVM and host process metrics (HeapMemoryMax, process counts, zabbix process metrics) appear frequently in the anomaly rows.

Top KPI patterns among MG01/MG02 metric anomalies (top 10 by occurrence):
- JVM-Memory_7779_JVM_Memory_HeapMemoryMax (most frequent)
- OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount
- OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies
- OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
- OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT
- JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed
- OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc
- OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem
- OSLinux-OSLinux_MEMORY_MEMORY_CacheMem
- OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT

Interpretation and recommended next steps:
- MG01: investigate disk IO on sda (DSKRead/DSKTps/DSKBps) and CPU WIO spikes — these exceed p95 by >50% and could explain service slowdowns.
- MG02: check disk IO spikes on sda as well (similar but slightly lower severity).
- Frequent anomalies in JVM heap and process/zabbix metrics suggest checking JVM memory and zabbix process behavior for possible resource issues or misconfigurations.
- Correlate these metric breaches with trace spikes (earlier trace analysis showed MG nodes and dockerA nodes with severe trace duration spikes) and with logs around the earliest anomaly timestamps to find root cause (IO contention, CPU wait, or a problematic process).
- If you want, I can: (a) show the top 20 severe metric groups table exactly as rows, (b) display the mg_metric_anoms sample rows around the peak timestamps, or (c) correlate these metric anomalies with trace/log anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(                     timestamp cmdb_id                                           kpi_name         value component           p50           p95 anomaly_type component_aggr      p50_aggr      p95_aggr
0    2021-03-24 16:30:00+08:00    MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09      MG01  2.040136e+09  2.040136e+09         high           MG01  2.040136e+09  2.040136e+09
1    2021-03-24 16:30:00+08:00    MG01                    OSLinux-CPU_CPU-2_SingleCpuUtil  6.750900e+00      MG01  4.658100e+00  6.473580e+00         high           MG01  4.658100e+00  6.473580e+00
2    2021-03-24 16:30:00+08:00    MG01                    OSLinux-CPU_CPU-3_SingleCpuUtil  1.000000e+02      MG01  1.000000e+02  1.000000e+02         high           MG01  1.000000e+02  1.000000e+02
3    2021-03-24 16:30:00+08:00    MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps  0.000000e+00      MG01  0.000000e+00  0.000000e+00         high           MG01  0.000000e+00  0.000000e+00
4    2021-03-24 16:30:00+08:00    MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  0.000000e+00      MG01  0.000000e+00  0.000000e+00         high           MG01  0.000000e+00  0.000000e+00
...                        ...     ...                                                ...           ...       ...           ...           ...          ...            ...           ...           ...
1706 2021-03-24 17:00:00+08:00    MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00      MG02  1.000000e+00  1.000000e+00         high           MG02  1.000000e+00  1.000000e+00
1707 2021-03-24 17:00:00+08:00    MG02          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem  5.800000e+01      MG02  5.800000e+01  5.800000e+01         high           MG02  5.800000e+01  5.800000e+01
1708 2021-03-24 17:00:00+08:00    MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01      MG02  3.000000e-01  3.000000e-01         high           MG02  3.000000e-01  3.000000e-01
1709 2021-03-24 17:00:00+08:00    MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00      MG02  3.000000e+00  3.000000e+00         high           MG02  3.000000e+00  3.000000e+00
1710 2021-03-24 17:00:00+08:00    MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.775504e+06      MG02  7.774603e+06  7.775415e+06         high           MG02  7.774603e+06  7.775415e+06

[1711 rows x 11 columns],     component                                           kpi_name  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window            p50            p95  severity_pct
59       MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                        1   2021-03-24 16:59:00+0800             6.000000       0.000000       2.500000      1.400000
19       MG01                             OSLinux-CPU_CPU_CPUWio                        2   2021-03-24 16:31:00+0800             3.765029       0.079065       1.858649      1.025680
82       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                        2   2021-03-24 16:44:00+0800             1.000000       0.000000       0.500000      1.000000
163      MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps                        1   2021-03-24 16:34:00+0800             6.000000       0.000000       3.200000      0.875000
168      MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps                        1   2021-03-24 16:30:00+0800            14.000000       0.000000       7.500000      0.866667
167      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...                        1   2021-03-24 16:30:00+0800           188.000000      10.000000     109.250000      0.720824
123      MG02                             OSLinux-CPU_CPU_CPUWio                        2   2021-03-24 16:30:00+0800             1.410500       0.033290       0.873660      0.614472
95       MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                       31   2021-03-24 16:30:00+0800             0.400000       0.300000       0.300000      0.333333
60       MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                        1   2021-03-24 16:51:00+0800             0.300000       0.000000       0.230000      0.304348
173      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...                        1   2021-03-24 16:47:00+0800             0.300000       0.000000       0.230000      0.304348
120      MG02                            OSLinux-CPU_CPU_CPULoad                        2   2021-03-24 16:33:00+0800             1.700000       1.140000       1.400000      0.214286
15       MG01                         OSLinux-CPU_CPU_CPUCpuUtil                        1   2021-03-24 16:48:00+0800            32.350100      28.388400      29.318180      0.103414
108      MG02          JVM-Operating System_7779_JVM_JVM_CPULoad                        1   2021-03-24 16:48:00+0800             3.072200       2.241750       2.822640      0.088414
16       MG01                            OSLinux-CPU_CPU_CPULoad                        2   2021-03-24 16:57:00+0800             1.920000       1.200000       1.765000      0.087819
170      MG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite                        1   2021-03-24 16:59:00+0800            15.600000      10.666667      14.386690      0.084336
4        MG01          JVM-Operating System_7779_JVM_JVM_CPULoad                        1   2021-03-24 16:48:00+0800             3.135800       2.262100       2.905670      0.079200
7        MG01                    OSLinux-CPU_CPU-0_SingleCpuUtil                        1   2021-03-24 16:32:00+0800             7.159500       4.999200       6.654330      0.075916
111      MG02                    OSLinux-CPU_CPU-0_SingleCpuUtil                        1   2021-03-24 16:37:00+0800             6.909800       3.905400       6.431135      0.074429
169      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps                        1   2021-03-24 16:59:00+0800             1.100000       0.850000       1.030000      0.067961
191      MG02    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec                        1   2021-03-24 16:55:00+0800        806438.000000  572761.000000  756315.200000      0.066272,   component  number_of_severe_kpis
0      MG01                      5
1      MG02                      2,                                              kpi_name  count
0            JVM-Memory_7779_JVM_Memory_HeapMemoryMax     62
96         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     62
92      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     62
95  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     62
81     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     62
3         JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed     45
78       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc     41
93          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem     39
76             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem     39
82       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     32)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse kernel variables: metric_anoms_window, metric_aggr
tz = pytz.timezone('Asia/Shanghai')

# Prepare source
df = metric_anoms_window.copy()

# Normalize component column name
if 'cmdb_id' not in df.columns and 'component' in df.columns:
    df = df.rename(columns={'component':'cmdb_id'})

# Ensure timestamp is timezone-aware in Asia/Shanghai
if 'timestamp' in df.columns:
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        if df['timestamp'].dt.tz is None:
            df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert(tz)
        else:
            df['timestamp'] = df['timestamp'].dt.tz_convert(tz)

# Filter to dockerA2 and dockerA1
dockerA_metric_anoms = df[df['cmdb_id'].isin(['dockerA2','dockerA1'])].copy()

# Join thresholds from metric_aggr (do NOT recompute)
ma = metric_aggr.copy()
if 'component' not in ma.columns and 'cmdb_id' in ma.columns:
    ma = ma.rename(columns={'cmdb_id':'component'})
ma_sel = ma[['component','kpi_name','p50','p95']].drop_duplicates()

dockerA_metric_anoms = dockerA_metric_anoms.merge(
    ma_sel,
    left_on=['cmdb_id','kpi_name'],
    right_on=['component','kpi_name'],
    how='left',
    suffixes=('','_aggr')
)

# Ensure numeric
dockerA_metric_anoms['value'] = pd.to_numeric(dockerA_metric_anoms.get('value', pd.Series(dtype=float)), errors='coerce')
dockerA_metric_anoms['p95'] = pd.to_numeric(dockerA_metric_anoms['p95'], errors='coerce')
dockerA_metric_anoms['p50'] = pd.to_numeric(dockerA_metric_anoms['p50'], errors='coerce')

# Aggregate grouped summary by (component, kpi_name)
grouped = (dockerA_metric_anoms
           .groupby(['cmdb_id','kpi_name'], dropna=False)
           .agg(anomaly_count_in_window=('value','count'),
                earliest_anomaly_timestamp=('timestamp','min'),
                max_value_in_window=('value','max'),
                p50=('p50','first'),
                p95=('p95','first'))
           .reset_index()
          )

# Compute severity_pct where p95 > 0
grouped['severity_pct'] = pd.NA
mask = grouped['p95'] > 0
grouped.loc[mask, 'severity_pct'] = (grouped.loc[mask, 'max_value_in_window'] - grouped.loc[mask, 'p95']) / grouped.loc[mask, 'p95']

# Determine if group has any 'high' anomaly rows (to define severe)
high_flags = (dockerA_metric_anoms[dockerA_metric_anoms.get('anomaly_type') == 'high']
              .groupby(['cmdb_id','kpi_name'])
              .size()
              .reset_index()[['cmdb_id','kpi_name']])
high_flags['has_high'] = True
grouped = grouped.merge(high_flags, on=['cmdb_id','kpi_name'], how='left')
grouped['has_high'] = grouped['has_high'].fillna(False)

# Mark severe groups: has_high == True and severity_pct > 0.5
grouped['is_severe'] = False
sev_mask = grouped['severity_pct'].notna() & (grouped['severity_pct'] > 0.5) & (grouped['has_high'] == True)
grouped.loc[sev_mask, 'is_severe'] = True

# Format earliest_anomaly_timestamp for display
grouped['earliest_anomaly_timestamp'] = pd.to_datetime(grouped['earliest_anomaly_timestamp']).dt.tz_convert(tz)
grouped['earliest_anomaly_timestamp_str'] = grouped['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Sort by severity_pct desc then anomaly_count_in_window desc and take top 20
grouped_sorted = grouped.sort_values(by=['severity_pct','anomaly_count_in_window'], ascending=[False, False])
display_cols = ['cmdb_id','kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp_str',
                'max_value_in_window','p50','p95','severity_pct','is_severe']
compact_top20 = (grouped_sorted.head(20)[display_cols]
                 .rename(columns={'cmdb_id':'component','earliest_anomaly_timestamp_str':'earliest_anomaly_timestamp'}))

# Round numeric columns for compactness
for col in ['max_value_in_window','p50','p95','severity_pct']:
    if col in compact_top20.columns:
        compact_top20[col] = pd.to_numeric(compact_top20[col], errors='coerce').round(6)

# (a) number of distinct severe KPIs for dockerA2 and dockerA1
per_component_counts = (grouped[grouped['is_severe']]
                        .groupby('cmdb_id', dropna=False)
                        .agg(number_of_severe_kpis=('kpi_name','nunique'))
                        .reindex(['dockerA2','dockerA1'], fill_value=0)
                        .reset_index()
                        .rename(columns={'cmdb_id':'component'}))

# (b) overall earliest severe anomaly timestamp among dockerA1/dockerA2 severe groups
if grouped[grouped['is_severe']].shape[0] > 0:
    earliest_severe_ts = pd.to_datetime(grouped[grouped['is_severe']]['earliest_anomaly_timestamp']).min().tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
else:
    earliest_severe_ts = None

# Final outputs: store full filtered rows in dockerA_metric_anoms (already set) and display compact table + summaries
dockerA_metric_anoms, compact_top20.reset_index(drop=True), per_component_counts, earliest_severe_ts

```
Out[6]:
```
```python
# Summarize dockerA1/dockerA2 metric anomaly findings in plain English using kernel variables
import pandas as pd

# Safe access to kernel-stored variables
rows_count = len(dockerA_metric_anoms) if 'dockerA_metric_anoms' in globals() else 0
grouped_df = grouped_sorted if 'grouped_sorted' in globals() else (pd.DataFrame() if 'dockerA_metric_anoms' in globals() else pd.DataFrame())
per_comp = per_component_counts if 'per_component_counts' in globals() else pd.DataFrame()
# Compute severe counts per component (fallback to per_comp if available)
if not per_comp.empty:
    dockerA2_severe = int(per_comp.loc[per_comp['component']=='dockerA2','number_of_severe_kpis'].iat[0]) if 'dockerA2' in per_comp['component'].values else 0
    dockerA1_severe = int(per_comp.loc[per_comp['component']=='dockerA1','number_of_severe_kpis'].iat[0]) if 'dockerA1' in per_comp['component'].values else 0
else:
    # Fallback: infer from grouped_df using 'is_severe'
    if not grouped_df.empty and 'is_severe' in grouped_df.columns:
        dockerA2_severe = int(grouped_df[(grouped_df['component']=='dockerA2') & (grouped_df['is_severe']==True)]['kpi_name'].nunique())
        dockerA1_severe = int(grouped_df[(grouped_df['component']=='dockerA1') & (grouped_df['is_severe']==True)]['kpi_name'].nunique())
    else:
        dockerA2_severe = 0
        dockerA1_severe = 0

# Earliest severe anomaly timestamp among dockerA1/dockerA2 (if any severe)
earliest_severe = None
if not grouped_df.empty and 'is_severe' in grouped_df.columns and grouped_df['is_severe'].any():
    try:
        tsmin = pd.to_datetime(grouped_df[grouped_df['is_severe']==True]['earliest_anomaly_timestamp']).min()
        # Format as string if not null
        earliest_severe = tsmin.strftime('%Y-%m-%d %H:%M:%S%z') if not pd.isna(tsmin) else None
    except Exception:
        earliest_severe = None

# Count how many groups have p95 == 0 or missing (these produce undefined/NaN severity_pct)
p95_zero_or_na = 0
if not grouped_df.empty and 'p95' in grouped_df.columns:
    p95_zero_or_na = int(((grouped_df['p95'] == 0) | (grouped_df['p95'].isna())).sum())

summary = (
    f"dockerA1/dockerA2 metric anomaly summary (window 2021-03-24 16:30–17:00 UTC+8):\n"
    f"- Total metric anomaly rows for dockerA1/dockerA2: {rows_count}\n"
    f"- Distinct severe KPIs (severity_pct > 0.5 and high): dockerA2 = {dockerA2_severe}, dockerA1 = {dockerA1_severe}\n"
    f"- Earliest severe anomaly timestamp among dockerA1/dockerA2: {earliest_severe}\n"
    f"- Note: {p95_zero_or_na} grouped KPI(s) for these components have p95 == 0 or missing, producing undefined severity_pct (cannot evaluate relative breach).\n\n"
    "Interpretation:\n"
    "- No KPIs met the strict severe criteria (severity_pct > 0.5 with high-sided anomalies) for dockerA1/dockerA2 in this window.\n"
    "- Many container-related metrics exist, but most either have p95==0 (threshold undefined) or only small relative breaches vs p95.\n"
    "\nRecommended next steps:\n"
    "1) Inspect container KPIs with p95==0 to determine if thresholds are meaningful (avoid false positives).\n"
    "2) Review the particular container metrics with the largest absolute values/tiny relative increases (see stored dockerA_metric_anoms for details).\n"
    "3) Correlate container metrics with the trace-level severe spikes (dockerA/destination traces) already identified to find root cause.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    timestamp   cmdb_id                                           kpi_name         value component           p50           p95 anomaly_type component_aggr      p50_aggr      p95_aggr
0   2021-03-24 16:30:00+08:00  dockerA2  Container-DOCKER_CONTAINER_3bf443a64876--bcou-...  0.000000e+00  dockerA2  0.000000e+00  0.000000e+00         high       dockerA2  0.000000e+00  0.000000e+00
1   2021-03-24 16:30:00+08:00  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...  0.000000e+00  dockerA2  0.000000e+00  0.000000e+00         high       dockerA2  0.000000e+00  0.000000e+00
2   2021-03-24 16:30:00+08:00  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...  0.000000e+00  dockerA2  0.000000e+00  0.000000e+00         high       dockerA2  0.000000e+00  0.000000e+00
3   2021-03-24 16:30:00+08:00  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...  0.000000e+00  dockerA2  0.000000e+00  0.000000e+00         high       dockerA2  0.000000e+00  0.000000e+00
4   2021-03-24 16:30:00+08:00  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...  0.000000e+00  dockerA2  0.000000e+00  0.000000e+00         high       dockerA2  0.000000e+00  0.000000e+00
..                        ...       ...                                                ...           ...       ...           ...           ...          ...            ...           ...           ...
102 2021-03-24 16:59:00+08:00  dockerA2  Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...  1.978343e+09  dockerA2  1.976275e+09  1.977938e+09         high       dockerA2  1.976275e+09  1.977938e+09
103 2021-03-24 16:59:00+08:00  dockerA2  Container-DOCKER_CONTAINER_3bf443a64876--bcou-...  0.000000e+00  dockerA2  0.000000e+00  0.000000e+00         high       dockerA2  0.000000e+00  0.000000e+00
104 2021-03-24 16:59:00+08:00  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...  0.000000e+00  dockerA2  0.000000e+00  0.000000e+00         high       dockerA2  0.000000e+00  0.000000e+00
105 2021-03-24 17:00:00+08:00  dockerA2  Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-...  0.000000e+00  dockerA2  0.000000e+00  0.000000e+00         high       dockerA2  0.000000e+00  0.000000e+00
106 2021-03-24 17:00:00+08:00  dockerA2  Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-...  0.000000e+00  dockerA2  0.000000e+00  0.000000e+00         high       dockerA2  0.000000e+00  0.000000e+00

[107 rows x 11 columns],    component                                           kpi_name  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window           p50           p95  severity_pct  is_severe
0   dockerA2  Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...                        1   2021-03-24 16:41:00+0800         6.300000e-01  4.500000e-01  6.060000e-01      0.039604      False
1   dockerA2  Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...                        1   2021-03-24 16:53:00+0800         3.772570e+05  2.897310e+05  3.637060e+05      0.037258      False
2   dockerA2  Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...                        1   2021-03-24 16:55:00+0800         1.947104e+06  1.751720e+06  1.910042e+06      0.019404      False
3   dockerA2  Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...                        1   2021-03-24 16:59:00+0800         1.978343e+09  1.976275e+09  1.977938e+09      0.000205      False
4   dockerA2  Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...                        4   2021-03-24 16:31:00+0800         6.003000e+01  6.003000e+01  6.003000e+01      0.000000      False
5   dockerA2  Container-DOCKER_CONTAINER_2c2336e2994f--bcou-...                        1   2021-03-24 16:37:00+0800         3.072000e+09  3.072000e+09  3.072000e+09      0.000000      False
6   dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...                       12   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
7   dockerA2  Container-DOCKER_CONTAINER_3bf443a64876--bcou-...                       10   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
8   dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...                       10   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
9   dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...                       10   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
10  dockerA2  Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-...                        9   2021-03-24 16:33:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
11  dockerA2  Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-...                        9   2021-03-24 16:33:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
12  dockerA2  Container-DOCKER_CONTAINER_3bf443a64876--bcou-...                        7   2021-03-24 16:33:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
13  dockerA2  Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-...                        7   2021-03-24 16:39:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
14  dockerA2  Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-...                        7   2021-03-24 16:39:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
15  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...                        6   2021-03-24 16:30:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
16  dockerA2  Container-DOCKER_CONTAINER_3bf443a64876--bcou-...                        4   2021-03-24 16:35:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
17  dockerA2  Container-DOCKER_CONTAINER_3bf443a64876--bcou-...                        4   2021-03-24 16:42:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
18  dockerA2  Container-DOCKER_CONTAINER_76d31070b844--bcou-...                        2   2021-03-24 16:32:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False
19  dockerA2  Container-DOCKER_CONTAINER_69b53a78b2eb--bcou-...                        1   2021-03-24 16:50:00+0800         0.000000e+00  0.000000e+00  0.000000e+00           NaN      False,   component  number_of_severe_kpis
0  dockerA2                      0
1  dockerA1                      0, None)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')
components = ['MG02','MG01','dockerA2','dockerA1']

# Helper to normalize dataframe: drop duplicate columns and ensure 'component' and 'kpi_name' exist
def normalize_df(df):
    df = df.copy()
    # drop duplicated column labels if any
    df = df.loc[:, ~df.columns.duplicated()]
    # ensure component column
    if 'component' not in df.columns and 'cmdb_id' in df.columns:
        df = df.rename(columns={'cmdb_id':'component'})
    if 'component' not in df.columns and 'cmdb_id' in df.columns:
        df['component'] = df['cmdb_id']
    # ensure kpi/trace name column
    if 'kpi_name' not in df.columns and 'trace_name' in df.columns:
        df = df.rename(columns={'trace_name':'kpi_name'})
    if 'kpi_name' not in df.columns and 'trace_name' not in df.columns and 'kpi_name' in df.columns:
        pass
    return df

# Helper to coerce timestamp series to tz-aware Asia/Shanghai
def ensure_ts_series(s):
    s = pd.to_datetime(s, utc=True, errors='coerce')
    return s.dt.tz_convert(tz)

cols = ['component','kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window','p50','p95','severity_pct','source']
severe_rows = []

# Process filtered_anoms_severe (group-level) if present
if 'filtered_anoms_severe' in globals():
    df = normalize_df(filtered_anoms_severe)
    if {'anomaly_type','severity_pct','component'}.issubset(df.columns):
        df['severity_pct'] = pd.to_numeric(df['severity_pct'], errors='coerce')
        df2 = df[(df['anomaly_type']=='high') & (df['severity_pct']>0.5) & (df['component'].isin(components))].copy()
        if not df2.empty:
            df2['earliest_anomaly_timestamp'] = ensure_ts_series(df2['earliest_anomaly_timestamp'])
            for _, r in df2.iterrows():
                severe_rows.append([
                    r['component'],
                    r.get('kpi_name', None),
                    int(r.get('anomaly_count_in_window', 0)),
                    r['earliest_anomaly_timestamp'],
                    float(r.get('max_value_in_window', pd.NA)) if pd.notna(r.get('max_value_in_window', None)) else pd.NA,
                    float(r.get('p50', pd.NA)) if pd.notna(r.get('p50', None)) else pd.NA,
                    float(r.get('p95', pd.NA)) if pd.notna(r.get('p95', None)) else pd.NA,
                    float(r.get('severity_pct', pd.NA)) if pd.notna(r.get('severity_pct', None)) else pd.NA,
                    'filtered_anoms_severe'
                ])

# Process trace_severe_details (group-level) if present
if 'trace_severe_details' in globals():
    df = normalize_df(trace_severe_details)
    if 'severity_pct' in df.columns and 'component' in df.columns:
        df['severity_pct'] = pd.to_numeric(df['severity_pct'], errors='coerce')
        df2 = df[(df['severity_pct']>0.5) & (df['component'].isin(components))].copy()
        if not df2.empty:
            df2['earliest_anomaly_timestamp'] = ensure_ts_series(df2['earliest_anomaly_timestamp'])
            for _, r in df2.iterrows():
                severe_rows.append([
                    r['component'],
                    r.get('trace_name', r.get('kpi_name', None)),
                    int(r.get('anomaly_count_in_window', 0)),
                    r['earliest_anomaly_timestamp'],
                    float(r.get('max_value_in_window', pd.NA)) if pd.notna(r.get('max_value_in_window', None)) else pd.NA,
                    float(r.get('p50', pd.NA)) if pd.notna(r.get('p50', None)) else pd.NA,
                    float(r.get('p95', pd.NA)) if pd.notna(r.get('p95', None)) else pd.NA,
                    float(r.get('severity_pct', pd.NA)) if pd.notna(r.get('severity_pct', None)) else pd.NA,
                    'trace_severe_details'
                ])

# Process mg_metric_anoms if present (per-row): aggregate and filter severe groups
if 'mg_metric_anoms' in globals():
    df = normalize_df(mg_metric_anoms)
    if {'p95','value','component'}.issubset(df.columns):
        # group
        grp = (df[df['component'].isin(components)]
               .groupby(['component','kpi_name'], dropna=False)
               .agg(anomaly_count_in_window=('value','count'),
                    earliest_anomaly_timestamp=('timestamp','min'),
                    max_value_in_window=('value','max'),
                    p50=('p50','first'),
                    p95=('p95','first'))
               .reset_index())
        # determine if any high anomaly exists for the group
        if 'anomaly_type' in df.columns:
            highs = (df[df['anomaly_type']=='high']
                     .groupby(['component','kpi_name']).size().reset_index()[['component','kpi_name']])
            highs['has_high'] = True
            grp = grp.merge(highs, on=['component','kpi_name'], how='left')
            grp['has_high'] = grp['has_high'].fillna(False)
        else:
            grp['has_high'] = False
        grp['p95'] = pd.to_numeric(grp['p95'], errors='coerce')
        grp['max_value_in_window'] = pd.to_numeric(grp['max_value_in_window'], errors='coerce')
        mask = grp['p95'] > 0
        grp.loc[mask, 'severity_pct'] = (grp.loc[mask, 'max_value_in_window'] - grp.loc[mask, 'p95']) / grp.loc[mask, 'p95']
        grp2 = grp[(grp['has_high']==True) & (grp['severity_pct']>0.5)].copy()
        if not grp2.empty:
            grp2['earliest_anomaly_timestamp'] = ensure_ts_series(grp2['earliest_anomaly_timestamp'])
            for _, r in grp2.iterrows():
                severe_rows.append([
                    r['component'],
                    r['kpi_name'],
                    int(r['anomaly_count_in_window']),
                    r['earliest_anomaly_timestamp'],
                    float(r['max_value_in_window']) if pd.notna(r['max_value_in_window']) else pd.NA,
                    float(r['p50']) if pd.notna(r['p50']) else pd.NA,
                    float(r['p95']) if pd.notna(r['p95']) else pd.NA,
                    float(r['severity_pct']) if pd.notna(r['severity_pct']) else pd.NA,
                    'mg_metric_anoms'
                ])

# Process dockerA_metric_anoms similarly if present
if 'dockerA_metric_anoms' in globals():
    df = normalize_df(dockerA_metric_anoms)
    if {'p95','value','component'}.issubset(df.columns):
        grp = (df[df['component'].isin(components)]
               .groupby(['component','kpi_name'], dropna=False)
               .agg(anomaly_count_in_window=('value','count'),
                    earliest_anomaly_timestamp=('timestamp','min'),
                    max_value_in_window=('value','max'),
                    p50=('p50','first'),
                    p95=('p95','first'))
               .reset_index())
        if 'anomaly_type' in df.columns:
            highs = (df[df['anomaly_type']=='high']
                     .groupby(['component','kpi_name']).size().reset_index()[['component','kpi_name']])
            highs['has_high'] = True
            grp = grp.merge(highs, on=['component','kpi_name'], how='left')
            grp['has_high'] = grp['has_high'].fillna(False)
        else:
            grp['has_high'] = False
        grp['p95'] = pd.to_numeric(grp['p95'], errors='coerce')
        grp['max_value_in_window'] = pd.to_numeric(grp['max_value_in_window'], errors='coerce')
        mask = grp['p95'] > 0
        grp.loc[mask, 'severity_pct'] = (grp.loc[mask, 'max_value_in_window'] - grp.loc[mask, 'p95']) / grp.loc[mask, 'p95']
        grp2 = grp[(grp['has_high']==True) & (grp['severity_pct']>0.5)].copy()
        if not grp2.empty:
            grp2['earliest_anomaly_timestamp'] = ensure_ts_series(grp2['earliest_anomaly_timestamp'])
            for _, r in grp2.iterrows():
                severe_rows.append([
                    r['component'],
                    r['kpi_name'],
                    int(r['anomaly_count_in_window']),
                    r['earliest_anomaly_timestamp'],
                    float(r['max_value_in_window']) if pd.notna(r['max_value_in_window']) else pd.NA,
                    float(r['p50']) if pd.notna(r['p50']) else pd.NA,
                    float(r['p95']) if pd.notna(r['p95']) else pd.NA,
                    float(r['severity_pct']) if pd.notna(r['severity_pct']) else pd.NA,
                    'dockerA_metric_anoms'
                ])

# Assemble severe DataFrame
severe_df = pd.DataFrame(severe_rows, columns=cols)
if not severe_df.empty:
    severe_df['earliest_anomaly_timestamp'] = pd.to_datetime(severe_df['earliest_anomaly_timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
    severe_df['severity_pct'] = pd.to_numeric(severe_df['severity_pct'], errors='coerce')
    # Aggregate per component+kpi: earliest timestamp and max severity_pct
    agg = (severe_df
           .groupby(['component','kpi_name'], dropna=False)
           .agg(anomaly_count_in_window=('anomaly_count_in_window','sum'),
                earliest_anomaly_timestamp=('earliest_anomaly_timestamp','min'),
                max_value_in_window=('max_value_in_window','max'),
                p50=('p50','first'),
                p95=('p95','first'),
                max_severity_pct=('severity_pct','max'))
           .reset_index())
else:
    agg = pd.DataFrame(columns=['component','kpi_name','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window','p50','p95','max_severity_pct'])

# Per-component summary for the four components
comp_summary = (agg[agg['component'].isin(components)]
                .groupby('component', dropna=False)
                .agg(earliest_severe_timestamp=('earliest_anomaly_timestamp','min'),
                     max_severity_pct=('max_severity_pct','max'),
                     severe_group_count=('kpi_name','nunique'))
                .reindex(components)  # ensure order and presence
                .reset_index()
               )

# Format timestamp and numeric
def fmt_ts(x):
    if pd.isna(x):
        return pd.NA
    try:
        return pd.to_datetime(x).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    except Exception:
        try:
            return pd.to_datetime(x, utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
        except Exception:
            return pd.NA

comp_summary['earliest_severe_timestamp'] = comp_summary['earliest_severe_timestamp'].apply(fmt_ts)
comp_summary['max_severity_pct'] = pd.to_numeric(comp_summary['max_severity_pct'], errors='coerce').round(6)
comp_summary['severe_group_count'] = comp_summary['severe_group_count'].fillna(0).astype(int)

# (a) overall earliest component
earliest_row = comp_summary.dropna(subset=['earliest_severe_timestamp']).sort_values('earliest_severe_timestamp').head(1)
if not earliest_row.empty:
    earliest_component = earliest_row.iloc[0]['component']
    earliest_timestamp = earliest_row.iloc[0]['earliest_severe_timestamp']
else:
    earliest_component = pd.NA
    earliest_timestamp = pd.NA

# (b) component with largest max_severity_pct
max_row = comp_summary.dropna(subset=['max_severity_pct']).sort_values('max_severity_pct', ascending=False).head(1)
if not max_row.empty:
    max_component = max_row.iloc[0]['component']
    max_severity_value = float(max_row.iloc[0]['max_severity_pct'])
else:
    max_component = pd.NA
    max_severity_value = pd.NA

# Final compact outputs: comp_summary (four components), earliest info, max info
comp_summary, (earliest_component, earliest_timestamp), (max_component, max_severity_value)

```
Out[7]:
```
Summary for components [MG02, MG01, dockerA2, dockerA1] (severe high-sided anomalies within 2021-03-24 16:30–17:00 UTC+8):

- Per-component compact results:
  - MG02 — earliest_severe_timestamp: 2021-03-24 16:30:00+0800; max_severity_pct: 3.549593; severe_group_count: 30
  - MG01 — earliest_severe_timestamp: 2021-03-24 16:31:00+0800; max_severity_pct: 4.922420; severe_group_count: 28
  - dockerA2 — earliest_severe_timestamp: 2021-03-24 16:30:00+0800; max_severity_pct: 59.906344; severe_group_count: 10
  - dockerA1 — earliest_severe_timestamp: 2021-03-24 16:31:00+0800; max_severity_pct: 4.127296; severe_group_count: 10

- (a) Component with the overall earliest severe timestamp: MG02 at 2021-03-24 16:30:00+0800.
- (b) Component with the largest max severity_pct: dockerA2 with severity_pct ≈ 59.906344.

Interpretation (brief): MG02 and dockerA2 show the earliest severe hits at 16:30; dockerA2 exhibits the single largest relative breach (very large spike vs its historical p95), while MG01 and MG02 have the most distinct severe groups overall.

The original code execution output of IPython Kernel is also provided below for reference:

(  component earliest_severe_timestamp  max_severity_pct  severe_group_count
0      MG02  2021-03-24 16:30:00+0800          3.549593                  30
1      MG01  2021-03-24 16:31:00+0800          4.922420                  28
2  dockerA2  2021-03-24 16:30:00+0800         59.906344                  10
3  dockerA1  2021-03-24 16:31:00+0800          4.127296                  10, ('MG02', '2021-03-24 16:30:00+0800'), ('dockerA2', 59.906344))```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# 1) MG02 disk I/O evidence from metric anomalies
# Prefer mg_metric_anoms if available, else use metric_anoms_window filtered for MG02
if 'mg_metric_anoms' in globals() and isinstance(mg_metric_anoms, pd.DataFrame):
    src_metrics = mg_metric_anoms.copy()
else:
    src_metrics = metric_anoms_window.copy() if 'metric_anoms_window' in globals() else pd.DataFrame()

# Normalize column names
if not src_metrics.empty:
    if 'cmdb_id' not in src_metrics.columns and 'component' in src_metrics.columns:
        src_metrics = src_metrics.rename(columns={'component':'cmdb_id'})
    if 'kpi_name' not in src_metrics.columns and 'kpi' in src_metrics.columns:
        src_metrics = src_metrics.rename(columns={'kpi':'kpi_name'})
    # Ensure timestamp tz-aware (Asia/Shanghai)
    if 'timestamp' in src_metrics.columns:
        try:
            if not pd.api.types.is_datetime64_any_dtype(src_metrics['timestamp']):
                src_metrics['timestamp'] = pd.to_datetime(src_metrics['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
            else:
                if src_metrics['timestamp'].dt.tz is None:
                    src_metrics['timestamp'] = pd.to_datetime(src_metrics['timestamp'], utc=True).dt.tz_convert(tz)
                else:
                    src_metrics['timestamp'] = src_metrics['timestamp'].dt.tz_convert(tz)
        except Exception:
            src_metrics['timestamp'] = pd.to_datetime(src_metrics['timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
    # Filter to MG02
    src_metrics_mg02 = src_metrics[src_metrics['cmdb_id']=='MG02'].copy()
    # Filter kpi_name contains any of the disk patterns (case-sensitive)
    disk_pattern = r'(DSKRead|DSKTps|DSKBps)'
    mg02_disk_evidence = src_metrics_mg02[src_metrics_mg02['kpi_name'].str.contains(disk_pattern, na=False)].copy()
    # Join to metric_aggr to get p50/p95 if not present or to ensure correctness
    if 'metric_aggr' in globals():
        ma = metric_aggr.copy()
        if 'component' not in ma.columns and 'cmdb_id' in ma.columns:
            ma = ma.rename(columns={'cmdb_id':'component'})
        ma_sel = ma[['component','kpi_name','p50','p95']].drop_duplicates()
        mg02_disk_evidence = mg02_disk_evidence.merge(ma_sel,
                                                      left_on=['cmdb_id','kpi_name'],
                                                      right_on=['component','kpi_name'],
                                                      how='left',
                                                      suffixes=('','_aggr'))
    # Ensure numeric conversions
    mg02_disk_evidence['value'] = pd.to_numeric(mg02_disk_evidence.get('value', pd.Series(dtype=float)), errors='coerce')
    mg02_disk_evidence['p95'] = pd.to_numeric(mg02_disk_evidence.get('p95', pd.Series(dtype=float)), errors='coerce')
    # Compute severity_pct per row when p95 > 0
    mg02_disk_evidence['severity_pct'] = np.nan
    mask_p95 = mg02_disk_evidence['p95'] > 0
    mg02_disk_evidence.loc[mask_p95, 'severity_pct'] = (mg02_disk_evidence.loc[mask_p95, 'value'] - mg02_disk_evidence.loc[mask_p95, 'p95']) / mg02_disk_evidence.loc[mask_p95, 'p95']
    # Sort by timestamp asc then severity_pct desc
    mg02_disk_evidence = mg02_disk_evidence.sort_values(by=['timestamp','severity_pct'], ascending=[True, False])
    # Compact table for display (top 20)
    mg02_disk_compact = (mg02_disk_evidence
                         .loc[:, ['timestamp','cmdb_id','kpi_name','value','p50','p95','severity_pct']]
                         .head(20)
                        )
else:
    mg02_disk_evidence = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','p50','p95','severity_pct'])
    mg02_disk_compact = mg02_disk_evidence.copy()

# 2) Trace evidence for MG02 <-> dockerA2 interactions and dockerA2 self traces
# Prefer trace_anoms_window; else use trace_severe_details aggregated (but we need per-row)
trace_src = None
if 'trace_anoms_window' in globals() and isinstance(trace_anoms_window, pd.DataFrame):
    trace_src = trace_anoms_window.copy()
elif 'trace_severe_details' in globals() and isinstance(trace_severe_details, pd.DataFrame):
    # trace_severe_details is aggregated; use it (will have p50/p95/severity_pct)
    trace_src = trace_severe_details.copy()

if trace_src is None:
    mg02_dockerA2_trace_evidence = pd.DataFrame(columns=['timestamp','component','trace_name','value','p50','p95','severity_pct'])
    mg02_trace_compact = mg02_dockerA2_trace_evidence.copy()
else:
    # Normalize columns
    if 'component' not in trace_src.columns and 'cmdb_id' in trace_src.columns:
        trace_src = trace_src.rename(columns={'cmdb_id':'component'})
    if 'trace_name' not in trace_src.columns and 'kpi_name' in trace_src.columns:
        trace_src = trace_src.rename(columns={'kpi_name':'trace_name'})
    # Ensure timestamp tz-aware
    if 'timestamp' in trace_src.columns:
        try:
            if not pd.api.types.is_datetime64_any_dtype(trace_src['timestamp']):
                trace_src['timestamp'] = pd.to_datetime(trace_src['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
            else:
                if trace_src['timestamp'].dt.tz is None:
                    trace_src['timestamp'] = pd.to_datetime(trace_src['timestamp'], utc=True).dt.tz_convert(tz)
                else:
                    trace_src['timestamp'] = trace_src['timestamp'].dt.tz_convert(tz)
        except Exception:
            trace_src['timestamp'] = pd.to_datetime(trace_src['timestamp'], utc=True, errors='coerce').dt.tz_convert(tz)
    # Filter component in {MG02, dockerA2} and trace_name contains one of the patterns
    patterns = ['trace.to_dockerA2','trace.from_MG02','trace.from_MG01','trace.to_MG02','trace.self']
    regex_pat = '(' + '|'.join([p.replace('.', r'\.') for p in patterns]) + ')'
    trace_filtered = trace_src[(trace_src['component'].isin(['MG02','dockerA2'])) & (trace_src['trace_name'].str.contains(regex_pat, na=False))].copy()
    # If using raw anomaly rows (trace_anoms_window), join to trace_aggr for p50/p95
    if 'trace_anoms_window' in globals() and trace_src is trace_anoms_window:
        if 'trace_aggr' in globals():
            ta = trace_aggr.copy()
            if 'component' not in ta.columns and 'cmdb_id' in ta.columns:
                ta = ta.rename(columns={'cmdb_id':'component'})
            ta_sel = ta[['component','kpi_name','p50','p95']].drop_duplicates()
            trace_filtered = trace_filtered.merge(ta_sel,
                                                  left_on=['component','trace_name'],
                                                  right_on=['component','kpi_name'],
                                                  how='left',
                                                  suffixes=('','_aggr'))
        # Ensure numeric
        trace_filtered['value'] = pd.to_numeric(trace_filtered.get('value', pd.Series(dtype=float)), errors='coerce')
        trace_filtered['p95'] = pd.to_numeric(trace_filtered.get('p95', pd.Series(dtype=float)), errors='coerce')
        # Compute severity_pct per row when p95 > 0
        trace_filtered['severity_pct'] = np.nan
        mask_p95_t = trace_filtered['p95'] > 0
        trace_filtered.loc[mask_p95_t, 'severity_pct'] = (trace_filtered.loc[mask_p95_t, 'value'] - trace_filtered.loc[mask_p95_t, 'p95']) / trace_filtered.loc[mask_p95_t, 'p95']
    else:
        # trace_filtered likely already has p50/p95/severity_pct (from trace_severe_details)
        # Ensure numeric and presence
        trace_filtered['value'] = pd.to_numeric(trace_filtered.get('value', pd.Series(dtype=float)), errors='coerce')
        trace_filtered['p95'] = pd.to_numeric(trace_filtered.get('p95', pd.Series(dtype=float)), errors='coerce')
        trace_filtered['severity_pct'] = pd.to_numeric(trace_filtered.get('severity_pct', pd.Series(dtype=float)), errors='coerce')
    # Sort by timestamp asc then severity_pct desc
    mg02_dockerA2_trace_evidence = trace_filtered.sort_values(by=['timestamp','severity_pct'], ascending=[True, False]).copy()
    # Compact table select columns and top 20
    select_cols = ['timestamp','component','trace_name','value','p50','p95','severity_pct']
    available_cols = [c for c in select_cols if c in mg02_dockerA2_trace_evidence.columns]
    mg02_trace_compact = mg02_dockerA2_trace_evidence.loc[:, available_cols].head(20)

# 3) Counts
mg02_disk_count = int(mg02_disk_evidence.shape[0]) if not mg02_disk_evidence.empty else 0
mg02_trace_count = int(mg02_dockerA2_trace_evidence.shape[0]) if ('mg02_dockerA2_trace_evidence' in globals() and not mg02_dockerA2_trace_evidence.empty) else 0

# Final compact outputs: mg02_disk_compact, mg02_trace_compact, and counts
mg02_disk_compact, mg02_trace_compact, (mg02_disk_count, mg02_trace_count)

```
Out[8]:
```
summary = (
    "Evidence summary (MG02 disk I/O and MG02<->dockerA2 trace spikes) within 2021-03-24 16:30–17:00 (UTC+8):\n\n"
    "MG02 disk I/O (metric anomaly rows found = 57):\n"
    "- Notable breaches at 16:30:\n"
    "  * OSLinux-...-sda_DSKTps = 14.0 (p95=7.50) → severity_pct ≈ 0.867\n"
    "  * OSLinux-...-sda_DSKRead = 188.0 (p95=109.25) → severity_pct ≈ 0.721\n"
    "- At 16:34 DSKBps = 6.0 (p95=3.20) → severity_pct ≈ 0.875\n"
    "- Many other disk KPIs show zeros or p95==0 (severity undefined) but the above rows demonstrate concrete MG02 disk I/O spikes beginning at 16:30.\n\n"
    "Trace evidence for MG02 <-> dockerA2 and dockerA2 self (trace rows found = 45):\n"
    "- Cluster of large trace-duration spikes at ~16:31:\n"
    "  * MG02 trace.self.duration_p95 ≈ 24.23 (p95≈14.65)\n"
    "  * MG02 trace.to_dockerA2.duration_p95 ≈ 16.42 (p95≈9.77)\n"
    "  * dockerA2 trace.from_MG02.duration_p95 ≈ 16.42 (p95≈9.77)\n"
    "  * dockerA2 trace.to_MG02.duration_p95 ≈ 25.00 (p95≈16.05)\n"
    "- dockerA2 also shows large row_count spikes later (examples: row_count=3047 at 16:46 vs p95≈2995.5; other elevated row_count values at 16:45–16:46).\n"
    "- These trace spikes (around 16:31) align closely with the MG02 disk-I/O spikes (around 16:30), indicating correlated I/O pressure and increased service latency between MG02 and dockerA2.\n\n"
    "Interpretation and recommended next steps:\n"
    "- The data shows MG02 experienced disk I/O spikes starting at 16:30, and trace-level latency between MG02 and dockerA2 peaked at ~16:31. dockerA2 also shows elevated row counts shortly after.\n"
    "- This temporal correlation suggests disk I/O contention on MG02 may have contributed to increased request/service latency for dockerA2-related flows.\n"
    "- Recommended actions: inspect MG02 host disk I/O and I/O wait around 16:30–16:32, check container activity on dockerA2 at the same times, review relevant logs (services, kernel, storage), and consider deeper investigation of the containers listed in the stored evidence tables (mg02_disk_evidence and mg02_dockerA2_trace_evidence).\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp cmdb_id                                           kpi_name  value   p50     p95  severity_pct
1  2021-03-24 16:30:00+08:00    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps   14.0   0.0    7.50      0.866667
0  2021-03-24 16:30:00+08:00    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  188.0  10.0  109.25      0.720824
2  2021-03-24 16:30:00+08:00    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...    8.0   8.0    8.00      0.000000
3  2021-03-24 16:30:00+08:00    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    0.0   0.0    0.00           NaN
5  2021-03-24 16:31:00+08:00    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    3.0   0.0    3.00      0.000000
4  2021-03-24 16:31:00+08:00    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    0.0   0.0    0.00           NaN
6  2021-03-24 16:31:00+08:00    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.0   0.0    0.00           NaN
7  2021-03-24 16:32:00+08:00    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...    8.0   8.0    8.00      0.000000
8  2021-03-24 16:32:00+08:00    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    0.0   0.0    0.00           NaN
9  2021-03-24 16:33:00+08:00    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    0.0   0.0    0.00           NaN
10 2021-03-24 16:33:00+08:00    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.0   0.0    0.00           NaN
11 2021-03-24 16:34:00+08:00    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    6.0   0.0    3.20      0.875000
12 2021-03-24 16:35:00+08:00    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    0.0   0.0    0.00           NaN
13 2021-03-24 16:35:00+08:00    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.0   0.0    0.00           NaN
14 2021-03-24 16:36:00+08:00    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...    8.0   8.0    8.00      0.000000
15 2021-03-24 16:36:00+08:00    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    0.0   0.0    0.00           NaN
17 2021-03-24 16:37:00+08:00    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    3.0   0.0    3.00      0.000000
16 2021-03-24 16:37:00+08:00    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    0.0   0.0    0.00           NaN
18 2021-03-24 16:37:00+08:00    MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    0.0   0.0    0.00           NaN
19 2021-03-24 16:38:00+08:00    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...    8.0   8.0    8.00      0.000000,                     timestamp component                       trace_name        value          p50          p95  severity_pct
13  2021-03-24 16:30:00+08:00  dockerA2          trace.self.duration_p95     0.024100     0.007000     0.016550           NaN
83  2021-03-24 16:31:00+08:00      MG02         trace.self.duration_mean     3.907168     0.188010     2.603394           NaN
84  2021-03-24 16:31:00+08:00      MG02          trace.self.duration_p95    24.225750     1.074000    14.647575           NaN
87  2021-03-24 16:31:00+08:00      MG02  trace.to_dockerA2.duration_mean     1.940482     0.148876     1.204172           NaN
88  2021-03-24 16:31:00+08:00      MG02   trace.to_dockerA2.duration_p95    16.417000     1.028000     9.765425           NaN
140 2021-03-24 16:31:00+08:00  dockerA2    trace.from_MG01.duration_mean     1.782235     0.151203     1.176020           NaN
141 2021-03-24 16:31:00+08:00  dockerA2     trace.from_MG01.duration_p95    16.385300     1.028000    10.053400           NaN
142 2021-03-24 16:31:00+08:00  dockerA2    trace.from_MG02.duration_mean     1.940482     0.148876     1.204172           NaN
143 2021-03-24 16:31:00+08:00  dockerA2     trace.from_MG02.duration_p95    16.417000     1.028000     9.765425           NaN
144 2021-03-24 16:31:00+08:00  dockerA2         trace.self.duration_mean     0.274122     0.019634     0.172235           NaN
147 2021-03-24 16:31:00+08:00  dockerA2      trace.to_MG02.duration_mean     9.693581     0.063017     5.899713           NaN
148 2021-03-24 16:31:00+08:00  dockerA2       trace.to_MG02.duration_p95    25.003500     0.220925    16.049715           NaN
498 2021-03-24 16:32:00+08:00  dockerA2          trace.to_MG02.row_count    14.000000    35.000000    72.100000           NaN
517 2021-03-24 16:39:00+08:00  dockerA2          trace.to_MG02.row_count     5.000000    35.000000    72.100000           NaN
525 2021-03-24 16:40:00+08:00  dockerA2          trace.to_MG02.row_count     4.000000    35.000000    72.100000           NaN
245 2021-03-24 16:45:00+08:00  dockerA2          trace.to_MG02.row_count    82.000000    35.000000    72.100000           NaN
253 2021-03-24 16:46:00+08:00  dockerA2        trace.from_MG01.row_count   211.000000   166.000000   210.500000           NaN
254 2021-03-24 16:46:00+08:00  dockerA2             trace.self.row_count  3047.000000  2244.000000  2995.500000           NaN
318 2021-03-24 16:47:00+08:00      MG02         trace.self.duration_mean     6.898521     0.188010     2.603394           NaN
319 2021-03-24 16:47:00+08:00      MG02          trace.self.duration_p95    39.706800     1.074000    14.647575           NaN, (57, 45))```
```